<a href="https://colab.research.google.com/github/Chaitanya-G41/Plant_disease_detection/blob/main/notebooks/03_train_stage2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q timm
import sys
sys.path.append('/content/Plant_disease_detection')

Mounted at /content/drive


In [ ]:
import torch, os, random, numpy as np
import timm
from torch import nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms, datasets
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

In [ ]:
SAVE_DIR    = "/content/drive/MyDrive/PlantDiseaseProject/models/stage2"
LOCAL_TRAIN = "/content/data/train"
LOCAL_VAL   = "/content/data/val"
os.makedirs(SAVE_DIR, exist_ok=True)

CONFIG = {
    'train_dir'    : LOCAL_TRAIN,
    'val_dir'      : LOCAL_VAL,
    'save_dir'     : SAVE_DIR,
    'num_classes'  : 6,
    'batch_size'   : 16,
    'num_epochs'   : 20,
    'lr'           : 5e-5,
    'weight_decay' : 0.1,
    'warmup_epochs': 3,
    'patience'     : 7,
    'seed'         : 42,
    'num_workers'  : 4,
    'device'       : 'cuda' if torch.cuda.is_available() else 'cpu',
}

GUAVA_CLASSES = [
    'Guava_anthracnose', 'Guava_healthy', 'Guava_insect_bite',
    'Guava_multiple',    'Guava_scorch',  'Guava_yld'
]

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(CONFIG['seed'])
torch.backends.cudnn.benchmark = True
print(CONFIG)

{'train_dir': '/content/data/train', 'val_dir': '/content/data/val', 'save_dir': '/content/drive/MyDrive/PlantDiseaseProject/models/stage2', 'num_classes': 6, 'batch_size': 16, 'num_epochs': 20, 'lr': 5e-05, 'weight_decay': 0.1, 'warmup_epochs': 3, 'patience': 7, 'seed': 42, 'num_workers': 4, 'device': 'cuda'}


In [ ]:
import os, shutil, time
from concurrent.futures import ThreadPoolExecutor

DRIVE_TRAIN = "/content/drive/MyDrive/PlantDiseaseProject/dataset/train"
DRIVE_VAL   = "/content/drive/MyDrive/PlantDiseaseProject/dataset/val"

def copy_class(args):
    src_cls, dst_cls = args
    os.makedirs(dst_cls, exist_ok=True)
    for img in os.listdir(src_cls):
        src = os.path.join(src_cls, img)
        dst = os.path.join(dst_cls, img)
        if not os.path.exists(dst):
            shutil.copy2(src, dst)

def parallel_copy(src_dir, dst_dir, label):
    tasks = [
        (os.path.join(src_dir, cls), os.path.join(dst_dir, cls))
        for cls in os.listdir(src_dir)
        if os.path.isdir(os.path.join(src_dir, cls)) and cls.startswith("Guava")  # ← filter
    ]
    t0 = time.time()
    print(f"Copying {label} ({len(tasks)} classes): {[t[0].split('/')[-1] for t in tasks]}")
    with ThreadPoolExecutor(max_workers=8) as executor:
        executor.map(copy_class, tasks)
    print(f"Done in {time.time()-t0:.0f}s")

if not os.path.exists(LOCAL_TRAIN):
    parallel_copy(DRIVE_TRAIN, LOCAL_TRAIN, "train")
else:
    print("Train already local.")

if not os.path.exists(LOCAL_VAL):
    parallel_copy(DRIVE_VAL, LOCAL_VAL, "val")
else:
    print("Val already local.")

Copying train (6 classes): ['Guava_scorch', 'Guava_insect_bite', 'Guava_anthracnose', 'Guava_multiple', 'Guava_healthy', 'Guava_yld']
Done in 209s
Copying val (6 classes): ['Guava_yld', 'Guava_scorch', 'Guava_multiple', 'Guava_healthy', 'Guava_anthracnose', 'Guava_insect_bite']
Done in 50s


In [ ]:
def get_guava_transforms(split='train'):
    imagenet_mean = [0.485, 0.456, 0.406]
    imagenet_std  = [0.229, 0.224, 0.225]

    if split == 'train':
        return transforms.Compose([
            transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.ColorJitter(brightness=0.3, contrast=0.3,
                                   saturation=0.3, hue=0.1),
            # Gaussian Blur — simulates real-world camera blur
            transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0)),
            transforms.RandomRotation(30),
            transforms.RandomGrayscale(p=0.05),
            transforms.ToTensor(),
            transforms.Normalize(imagenet_mean, imagenet_std),
            transforms.RandomErasing(p=0.2),   # simulates occlusion
        ])
    else:
        return transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(imagenet_mean, imagenet_std),
        ])

def get_guava_dataloaders(train_dir, val_dir, batch_size, num_workers):
    # Filter only Guava classes
    train_full = datasets.ImageFolder(train_dir, transform=get_guava_transforms('train'))
    val_full   = datasets.ImageFolder(val_dir,   transform=get_guava_transforms('val'))

    guava_idx_train = [i for i, (_, label) in enumerate(train_full.samples)
                       if train_full.classes[label] in GUAVA_CLASSES]
    guava_idx_val   = [i for i, (_, label) in enumerate(val_full.samples)
                       if val_full.classes[label]   in GUAVA_CLASSES]

    train_dataset = torch.utils.data.Subset(train_full, guava_idx_train)
    val_dataset   = torch.utils.data.Subset(val_full,   guava_idx_val)

    # Weighted sampler for balance
    labels      = [train_full.samples[i][1] for i in guava_idx_train]
    class_count = np.bincount(labels)
    weights     = 1.0 / class_count[labels]
    sampler     = WeightedRandomSampler(weights, len(weights))

    train_loader = DataLoader(train_dataset, batch_size=batch_size,
                              sampler=sampler, num_workers=num_workers,
                              pin_memory=True, persistent_workers=True)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size,
                              shuffle=False,  num_workers=num_workers,
                              pin_memory=True, persistent_workers=True)

    # Remap class names to Guava only
    guava_classes  = sorted(GUAVA_CLASSES)
    idx_to_class   = {i: c for i, c in enumerate(guava_classes)}

    print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")
    print(f"Guava classes: {guava_classes}")
    return train_loader, val_loader, guava_classes, idx_to_class

train_loader, val_loader, class_names, idx_to_class = get_guava_dataloaders(
    CONFIG['train_dir'], CONFIG['val_dir'],
    CONFIG['batch_size'], CONFIG['num_workers']
)

Train batches: 300 | Val batches: 75
Guava classes: ['Guava_anthracnose', 'Guava_healthy', 'Guava_insect_bite', 'Guava_multiple', 'Guava_scorch', 'Guava_yld']


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [ ]:
# Load stage1 checkpoint
stage1_ckpt = torch.load(
    "/content/drive/MyDrive/PlantDiseaseProject/models/stage1/stage1_best.pth",
    map_location=CONFIG['device']
)

# Build model with 44 classes (same as stage1)
model = timm.create_model('deit_tiny_patch16_224', pretrained=False, num_classes=44)
model.load_state_dict(stage1_ckpt['model_state_dict'])

# Replace head with 6-class head for Guava
model.head = nn.Linear(model.head.in_features, CONFIG['num_classes'])

# Freeze blocks 0-9, train blocks 10-11 + norm + head
for name, param in model.named_parameters():
    param.requires_grad = False  # freeze all first

for name, param in model.named_parameters():
    if any(x in name for x in ['blocks.10', 'blocks.11', 'norm', 'head']):
        param.requires_grad = True  # unfreeze last 2 blocks + head

model = model.to(CONFIG['device'])

# Confirm trainable params
total   = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total:,}")
print(f"Trainable params: {trainable:,} ({100*trainable/total:.1f}%)")

Total params    : 5,525,574
Trainable params: 898,950 (16.3%)


In [ ]:
from torch.amp import GradScaler, autocast
import json

def train_stage2(model, train_loader, val_loader, config):
    optimizer = AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=config['lr'], weight_decay=config['weight_decay']
    )
    scheduler = CosineAnnealingLR(optimizer, T_max=config['num_epochs'])
    scaler    = GradScaler('cuda')
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    patience_ctr = 0
    history      = []

    for epoch in range(1, config['num_epochs'] + 1):
        # Warmup
        if epoch <= config['warmup_epochs']:
            for pg in optimizer.param_groups:
                pg['lr'] = config['lr'] * epoch / config['warmup_epochs']

        # ── Train ──
        model.train()
        train_loss, train_correct, total = 0, 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(config['device']), labels.to(config['device'])
            optimizer.zero_grad()
            with autocast('cuda'):
                out  = model(imgs)
                loss = criterion(out, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss    += loss.item() * imgs.size(0)
            train_correct += (out.argmax(1) == labels).sum().item()
            total         += imgs.size(0)

        train_loss /= total
        train_acc   = 100 * train_correct / total

        # ── Val ──
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(config['device']), labels.to(config['device'])
                with autocast('cuda'):
                    out  = model(imgs)
                    loss = criterion(out, labels)
                val_loss    += loss.item() * imgs.size(0)
                val_correct += (out.argmax(1) == labels).sum().item()
                val_total   += imgs.size(0)

        val_loss /= val_total
        val_acc   = 100 * val_correct / val_total

        if epoch > config['warmup_epochs']:
            scheduler.step()

        history.append({
            'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
            'train_acc': train_acc, 'val_acc': val_acc
        })

        print(f"Epoch {epoch:03d}/{config['num_epochs']} | "
              f"Train: {train_loss:.4f} / {train_acc:.2f}% | "
              f"Val: {val_loss:.4f} / {val_acc:.2f}%")

        # Save best
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'epoch'            : epoch,
                'model_state_dict' : model.state_dict(),
                'val_acc'          : val_acc,
                'class_names'      : class_names,
                'idx_to_class'     : idx_to_class,
            }, f"{config['save_dir']}/stage2_best.pth")
            print(f"  ✅ Saved best checkpoint (val_acc={val_acc:.2f}%)")
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= config['patience']:
                print(f"Early stopping at epoch {epoch}")
                break

    # Save history
    with open(f"{config['save_dir']}/stage2_history.json", 'w') as f:
        json.dump(history, f, indent=2)

    return history, best_val_acc

# ⚠️ Run Once Only
history, best_val_acc = train_stage2(model, train_loader, val_loader, CONFIG)
print(f"\nBest val accuracy: {best_val_acc:.2f}%")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch 001/20 | Train: 0.7360 / 80.00% | Val: 0.0532 / 100.00%
  ✅ Saved best checkpoint (val_acc=100.00%)
Epoch 002/20 | Train: 0.0425 / 99.21% | Val: 0.0080 / 100.00%
Epoch 003/20 | Train: 0.0264 / 99.29% | Val: 0.0032 / 100.00%
Epoch 004/20 | Train: 0.0196 / 99.54% | Val: 0.0016 / 100.00%
Epoch 005/20 | Train: 0.0260 / 99.23% | Val: 0.0016 / 100.00%
Epoch 006/20 | Train: 0.0189 / 99.42% | Val: 0.0011 / 100.00%
Epoch 007/20 | Train: 0.0159 / 99.62% | Val: 0.0008 / 100.00%
Epoch 008/20 | Train: 0.0116 / 99.65% | Val: 0.0005 / 100.00%
Early stopping at epoch 8

Best val accuracy: 100.00%


In [ ]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint = torch.load(f"{SAVE_DIR}/stage2_best.pth", map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
print(f"Loaded checkpoint from epoch {checkpoint['epoch']} | val_acc={checkpoint['val_acc']:.2f}%")

Loaded checkpoint from epoch 1 | val_acc=100.00%


In [ ]:
import json

# Reconstruct what we know from the training output
stage2_history = {
    "train_loss": [0.7360, 0.0425, 0.0264, 0.0196, 0.0260, None, None, None],
    "train_acc":  [80.00,  99.21,  99.29,  99.54,  99.23,  None, None, None],
    "val_loss":   [0.0532, 0.0080, 0.0032, 0.0016, 0.0016, None, None, None],
    "val_acc":    [100.0,  100.0,  100.0,  100.0,  100.0,  None, None, None],
    "best_epoch": 1,
    "best_val_acc": 100.0,
    "early_stopped_at": 8,
    "note": "Epochs 6-8 values not captured due to session reset"
}

with open(f"{SAVE_DIR}/stage2_history.json", "w") as f:
    json.dump(stage2_history, f, indent=2)
print("History saved.")

History saved.


In [ ]:
from sklearn.metrics import classification_report
import torch

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        outputs = model(imgs)
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(classification_report(all_labels, all_preds, target_names=checkpoint["class_names"]))

                   precision    recall  f1-score   support

Guava_anthracnose       1.00      1.00      1.00       200
    Guava_healthy       1.00      1.00      1.00       200
Guava_insect_bite       1.00      1.00      1.00       200
   Guava_multiple       1.00      1.00      1.00       200
     Guava_scorch       1.00      1.00      1.00       200
        Guava_yld       1.00      1.00      1.00       200

         accuracy                           1.00      1200
        macro avg       1.00      1.00      1.00      1200
     weighted avg       1.00      1.00      1.00      1200



In [ ]:
import glob
matches = glob.glob("/content/drive/MyDrive/PlantDiseaseProject/**/*.pth", recursive=True)
for m in matches:
    print(m)

/content/drive/MyDrive/PlantDiseaseProject/models/stage1/stage1_best.pth
/content/drive/MyDrive/PlantDiseaseProject/models/stage2/stage2_best.pth


In [ ]:
print("Stage 1 checkpoint keys:", stage1_ckpt.keys())

Stage 1 checkpoint keys: dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'val_acc', 'val_loss'])


In [ ]:
import os

s1_classes = sorted([
    cls for cls in os.listdir("/content/drive/MyDrive/PlantDiseaseProject/dataset/train")
    if os.path.isdir(os.path.join("/content/drive/MyDrive/PlantDiseaseProject/dataset/train", cls))
])
print(f"Found {len(s1_classes)} classes")
print(s1_classes)

Found 44 classes
['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Guava_anthracnose', 'Guava_healthy', 'Guava_insect_bite', 'Guava_multiple', 'Guava_scorch', 'Guava_yld', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight'

In [ ]:
stage2_ckpt = torch.load(f"{SAVE_DIR_S2}/stage2_best.pth", map_location=DEVICE)
model_s2 = timm.create_model("deit_tiny_patch16_224", pretrained=False, num_classes=6)
model_s2.load_state_dict(stage2_ckpt["model_state_dict"])
model_s2.to(DEVICE).eval()
s2_classes = stage2_ckpt["class_names"]

print("S2 classes:", s2_classes)

S2 classes: ['Guava_anthracnose', 'Guava_healthy', 'Guava_insect_bite', 'Guava_multiple', 'Guava_scorch', 'Guava_yld']


In [ ]:
code = '''import torch
import timm
import os
from torchvision import transforms
from PIL import Image

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SAVE_DIR_S1 = "/content/drive/MyDrive/PlantDiseaseProject/models/stage1"
SAVE_DIR_S2 = "/content/drive/MyDrive/PlantDiseaseProject/models/stage2"

# ── Load Stage 1 ──────────────────────────────────────────────
stage1_ckpt = torch.load(f"{SAVE_DIR_S1}/stage1_best.pth", map_location=DEVICE)
model_s1 = timm.create_model("deit_tiny_patch16_224", pretrained=False, num_classes=44)
model_s1.load_state_dict(stage1_ckpt["model_state_dict"])
model_s1.to(DEVICE).eval()

# ── Load Stage 2 ──────────────────────────────────────────────
stage2_ckpt = torch.load(f"{SAVE_DIR_S2}/stage2_best.pth", map_location=DEVICE)
model_s2 = timm.create_model("deit_tiny_patch16_224", pretrained=False, num_classes=6)
model_s2.load_state_dict(stage2_ckpt["model_state_dict"])
model_s2.to(DEVICE).eval()

# ── Class Names ───────────────────────────────────────────────
s1_classes = sorted([
    cls for cls in os.listdir("/content/drive/MyDrive/PlantDiseaseProject/dataset/train")
    if os.path.isdir(os.path.join("/content/drive/MyDrive/PlantDiseaseProject/dataset/train", cls))
])
s2_classes = stage2_ckpt["class_names"]
guava_indices = {i for i, c in enumerate(s1_classes) if c.startswith("Guava")}

# ── Transform ─────────────────────────────────────────────────
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# ── Inference Function ────────────────────────────────────────
def predict(image_path):
    img = Image.open(image_path).convert("RGB")
    tensor = transform(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        s1_out = model_s1(tensor)
        s1_probs = torch.softmax(s1_out, dim=1)
        s1_conf, s1_idx = s1_probs.max(dim=1)
        s1_idx = s1_idx.item()
        s1_conf = s1_conf.item()
        s1_label = s1_classes[s1_idx]

    if s1_idx in guava_indices:
        with torch.no_grad():
            s2_out = model_s2(tensor)
            s2_probs = torch.softmax(s2_out, dim=1)
            s2_conf, s2_idx = s2_probs.max(dim=1)
            s2_label = s2_classes[s2_idx.item()]
            s2_conf = s2_conf.item()

        print(f"[Stage 1] Guava detected ({s1_conf:.2%})")
        print(f"[Stage 2] {s2_label} ({s2_conf:.2%})")
        return {"plant": "Guava", "disease": s2_label, "confidence": s2_conf, "stage": 2}
    else:
        print(f"[Stage 1] {s1_label} ({s1_conf:.2%})")
        return {"plant": s1_label.split("___")[0], "disease": s1_label, "confidence": s1_conf, "stage": 1}
'''

save_path = "/content/drive/MyDrive/PlantDiseaseProject/inference.py"
with open(save_path, "w") as f:
    f.write(code)

print(f"Saved to {save_path}")

Saved to /content/drive/MyDrive/PlantDiseaseProject/inference.py


In [ ]:
print(os.path.exists("/content/drive/MyDrive/PlantDiseaseProject/inference.py"))  # True

True
